In [1]:
import os
import json
import random
import numpy as np
import torch

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
!git clone https://github.com/lxcentry/cu_konk_stip_ml_red_task.git

Cloning into 'cu_konk_stip_ml_red_task'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 34 (delta 3), reused 29 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 5.23 MiB | 20.38 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [3]:
!pip install -q -U transformers accelerate peft trl bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.7 MB/s eta 0:00:00


In [4]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen3-4B-Instruct-2507",
    local_dir="./qwen3_4b",
    local_dir_use_symlinks=False,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

'/content/qwen3_4b'

In [5]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

MODEL_NAME = "./qwen3_4b"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [6]:
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 132,120,576 || all params: 4,154,588,672 || trainable%: 3.1801


In [7]:
import json
from datasets import Dataset

examples = []

with open("cu_konk_stip_ml_red_task/ml-olympiad-red-task/data/kid_adult.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        messages = [
            {"role": "user", "content": row["prompt"]},
            {"role": "assistant", "content": row["kid"]},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        examples.append({"text": text})

train_dataset = Dataset.from_list(examples)

print(train_dataset[0])

{'text': '<|im_start|>user\nКак работает камера видеонаблюдения? Ответ: Она ловит свет через стеклышко, превращает его в электрические сигналы и сохраняет их как видео в памяти.<|im_end|>\n<|im_start|>assistant\nКамера работает как волшебный глаз. Она улавливает свет через свое стеклышко, переводит его в электрические сигналы и сохраняет как видео в памяти.<|im_end|>\n'}


In [8]:
from trl import SFTTrainer
import inspect

print(inspect.signature(SFTTrainer))

(model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallback] | None =

In [13]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./sft_model",

    learning_rate=2e-4,

    num_train_epochs=1,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=8,

    logging_steps=10,

    save_strategy="epoch",

    max_length=512,

    seed=42,

    bf16=True,
    fp16=False,
)

In [14]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [15]:
trainer.train()

Step,Training Loss
10,1.660399
20,1.287253
30,1.270499
40,1.201411
50,1.194139
60,1.146392
70,1.151661
80,1.133454
90,1.153325


TrainOutput(global_step=94, training_loss=1.238951804790091, metrics={'train_runtime': 1056.5148, 'train_samples_per_second': 1.409, 'train_steps_per_second': 0.089, 'total_flos': 4998357502393344.0, 'train_loss': 1.238951804790091, 'entropy': 1.1584913921356201, 'num_tokens': 204259.0, 'mean_token_accuracy': 0.7151740789413452, 'epoch': 1.0})

In [16]:
trainer.save_model("./sft_adapter")

tokenizer.save_pretrained("./sft_adapter")

('./sft_adapter/tokenizer_config.json',
 './sft_adapter/chat_template.jinja',
 './sft_adapter/tokenizer.json')

In [17]:
import json

test = []

with open("/content/cu_konk_stip_ml_red_task/ml-olympiad-red-task/data/public_test_style.jsonl", encoding="utf-8") as f:
    for line in f:
        test.append(json.loads(line))

print(test[0])

{'prompt': 'Почему продавцы в магазинах делают скидки и распродажи? Ответ: Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привлечь больше покупателей.', 'kid': 'Магазину нужно освободить полки, чтобы поставить новые игрушки и вещи. Поэтому старые товары продают дешевле, как будто кричат: «Забирайте скорее!». Это здорово помогает быстро всё разобрать и порадовать покупателей.', 'adult': 'Продавцы делают распродажи для ускорения оборачиваемости товаров и освобождения складских запасов перед закупкой новых коллекций. Маркетинговое снижение цены стимулирует покупательский спрос и позволяет конвертировать неликвидные активы в живые деньги, избегая замораживания капитала.', 'fact': 'Скидки и распродажи используются для ускорения сбыта остатков, расчистки складов под новые поступления и стимуляции спроса.'}


In [18]:
import joblib

style_clf = joblib.load("/content/cu_konk_stip_ml_red_task/ml-olympiad-red-task/metrics/style_clf.pkl")

print(type(style_clf))
print(style_clf)

<class 'dict'>
{'clf': LogisticRegression(C=4.0, max_iter=2000), 'vecs': (TfidfVectorizer(max_features=8000, min_df=2, ngram_range=(1, 2),
                sublinear_tf=True), TfidfVectorizer(analyzer='char_wb', max_features=8000, min_df=3,
                ngram_range=(2, 4), sublinear_tf=True))}


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking c

In [20]:
import joblib
from scipy.sparse import hstack

style = joblib.load("/content/cu_konk_stip_ml_red_task/ml-olympiad-red-task/metrics/style_clf.pkl")

clf = style["clf"]
word_vec, char_vec = style["vecs"]

In [22]:
from tqdm import tqdm

predictions = []

model.eval()

for sample in tqdm(test):

    messages = [
        {"role": "user", "content": sample["prompt"]}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=None,
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    predictions.append(
        {
            "prompt": sample["prompt"],
            "prediction": answer
        }
    )

  0%|          | 0/50 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 50/50 [09:36<00:00, 11.54s/it]


In [23]:
with open("predictions_sft.jsonl", "w", encoding="utf-8") as f:
    for row in predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [24]:
texts = [x["prediction"] for x in predictions]

In [25]:
X_word = word_vec.transform(texts)
X_char = char_vec.transform(texts)

X = hstack([X_word, X_char])

In [26]:
labels = clf.predict(X)

print(labels[:20])

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [27]:
print(clf.classes_)
print(clf.predict(X[:10]))

[0 1]
[1 1 1 1 1 1 1 1 1 1]


In [32]:
probs = clf.predict_proba(X)

P_simple = probs[:, 1].mean()

print(f"P_simple = {P_simple:.4f}")

P_simple = 0.9781
